In [56]:
import pandas as pd
import datetime as dt
import json

In [57]:
df = pd.read_csv('data.csv', sep=";")

In [58]:
df.head()

,Rank,Mark,Competitor,DOB,Nat,Pos,Venue,Date,Results Score,Mark [meters or seconds],Event,Wind,Sex
0,1,3:43.13,Hicham EL GUERROUJ,1974-09-14,MAR,1,"Stadio Olimpico, Roma (ITA)",1999-07-07,1292,223.1,One Mile,NaN,male
1,2,3:43.40,Noah NGENY,1978-11-02,KEN,2,"Stadio Olimpico, Roma (ITA)",1999-07-07,1288,223.4,One Mile,NaN,male
2,3,3:44.39,Noureddine MORCELI,1970-02-28,ALG,1,Rieti (ITA),1993-09-05,1275,224.3,One Mile,NaN,male
3,4,3:44.60,Hicham EL GUERROUJ,1974-09-14,MAR,1,Nice (FRA),1998-07-16,1272,224.6,One Mile,NaN,male
4,5,3:44.90,Hicham EL GUERROUJ,1974-09-14,MAR,1,Oslo (NOR),1997-07-04,1268,224.9,One Mile,NaN,male


# PRE-PROCESS DATASET

In [59]:
try:
    df["Date"] = df.apply(lambda row: dt.datetime.strptime(str(row["Date"]), "%Y-%m-%d"), axis=1)
except ValueError:
    print(f"Error: The date is not in the correct format.")
    

In [60]:
try:
    df["DOB"] = df.apply(lambda row: dt.datetime.strptime(str(row["DOB"]), "%Y-%m-%d") if pd.notna(row["DOB"]) else pd.NaT, axis=1)
except ValueError as e:
    print(f"Error: The date is not in the correct format.")
    print(e)

In [61]:
df["Age"] = df.apply(lambda row: (row["Date"] - row["DOB"]).days // 365 if pd.notna(row["DOB"]) else pd.NaT, axis=1)

In [62]:
df[df['Age'] < 14]

,Rank,Mark,Competitor,DOB,Nat,Pos,Venue,Date,Results Score,Mark [meters or seconds],Event,Wind,Sex,Age
12878,11113,19.91,Milton WILLIAMS,1979-11-10,USA,2,"Arlington, TX (USA)",1985-05-04,1116,19.9,Shot Put,NaN,male,5
34576,389,1:19:36,Aigars FADEJEVS,1975-12-27,URS,NaN,Naumburg (GER),1987-05-25,1200,4776.0,20 Kilometres Race Walk,NaN,male,11
150452,616,13:49,Hector PEREZ,2004-10-12,MEX,9,"Carlsbad, CA (USA)",1991-04-14,1037,829.0,5 Kilometres,NaN,male,-14
179472,7441,13:28.38,Pius Maluko MULI,1982-12-15,KEN,2,Americana (BRA),1996-03-16,1108,808.3,5000 Metres,NaN,male,13
245955,471,22:31,Anastasiya KOLCHINA,2005-05-11,RUS,2,Sochi (RUS),2018-02-18,1038,1351.0,5 Kilometres Race Walk,NaN,female,12
245981,500,22:34,Anastasiya KOLCHINA,2005-05-11,RUS,2.0,Sochi (RUS),2019-02-17,1035,1354.0,5 Kilometres Race Walk,NaN,female,13
246035,546,22:38,Sabina SLESAREVA,2005-06-27,RUS,3.0,Sochi (RUS),2019-02-17,1030,1358.0,5 Kilometres Race Walk,NaN,female,13
316866,9234,10:05.0h,Cynthia JEROP,1995-03-20,KEN,1,Nairobi (KEN),2006-07-01,1083,605.0,3000 Metres Steeplechase,NaN,female,11
318009,10377,10:08.1h,Beatrice Jepmwetich KIBOR,1992-12-10,KEN,2,Nairobi (KEN),2006-07-01,1076,608.1,3000 Metres Steeplechase,NaN,female,13
318010,10377,10:08.1h,Beatrice Jepmwetich KIBOR,1992-12-10,KEN,2,Kasarani (KEN),2006-07-01,1076,608.1,3000 Metres Steeplechase,NaN,female,13


virer les âges < 14

In [63]:
df["Age"] = df["Age"].apply(lambda row: row if pd.notna(row) and row >= 14 else pd.NaT)

## catégorisation cols

In [64]:
df["Event"].unique()

array(['One Mile', 'Shot Put', '2000 Metres Steeplechase', 'Discus Throw',
       '5000 Metres Race Walk', '5 Kilometres Race Walk', '10 Kilometres',
       '10000 Metres Race Walk', '20 Kilometres Race Walk',
       '15 Kilometres', '1500 Metres', '400 Metres Hurdles',
       '110 Metres Hurdles', '100 Metres', '600 Metres',
       '3000 Metres Steeplechase', 'Decathlon', '3000 Metres',
       'Two Miles', '10 Miles Road', '50 Kilometres Race Walk',
       'Pole Vault', 'Hammer Throw', '10000 Metres', '5 Kilometres',
       '20 Kilometres', '200 Metres', 'Triple Jump', '2000 Metres',
       '20000 Metres Race Walk', '5000 Metres', '1000 Metres',
       '400 Metres', '30 Kilometres Race Walk', '300 Metres', 'Marathon',
       '3000 Metres Race Walk', 'Javelin Throw', 'Half Marathon',
       'High Jump', '35 Kilometres Race Walk', '800 Metres',
       '10 Kilometres Race Walk', 'Long Jump', '100 Metres Hurdles',
       'Heptathlon'], dtype=object)

In [65]:
DIST_EVENTS = ["Shot Put", "Discus Throw", "Hammer Throw", "Javelin Throw", "Long Jump", "Triple Jump", "High Jump", "Pole Vault"]
TIME_EVENTS = ["One Mile", "2000 Metres Steeplechase", "5000 Metres Race Walk", "5 Kilometres Race Walk",
    "10 Kilometres", "10000 Metres Race Walk", "20 Kilometres Race Walk", "15 Kilometres", "1500 Metres",
    "400 Metres Hurdles", "110 Metres Hurdles", "100 Metres", "600 Metres", "3000 Metres Steeplechase",
    "3000 Metres", "Two Miles", "10 Miles Road", "50 Kilometres Race Walk", "10000 Metres", "5 Kilometres",
    "20 Kilometres", "200 Metres", "2000 Metres", "20000 Metres Race Walk", "5000 Metres", "1000 Metres",
    "400 Metres", "30 Kilometres Race Walk", "300 Metres", "Marathon", "3000 Metres Race Walk",
    "Half Marathon", "800 Metres", "10 Kilometres Race Walk", "100 Metres Hurdles", "35 Kilometres Race Walk",
    "Heptathlon", "Decathlon"]

# col pour checker si dist event ou pas

In [66]:
df["Dist_Event"] = df["Event"].apply(lambda x: 1 if x in DIST_EVENTS else 0)

In [67]:
df.head()

,Rank,Mark,Competitor,DOB,Nat,Pos,Venue,Date,Results Score,Mark [meters or seconds],Event,Wind,Sex,Age,Dist_Event
0,1,3:43.13,Hicham EL GUERROUJ,1974-09-14,MAR,1,"Stadio Olimpico, Roma (ITA)",1999-07-07,1292,223.1,One Mile,NaN,male,24,0
1,2,3:43.40,Noah NGENY,1978-11-02,KEN,2,"Stadio Olimpico, Roma (ITA)",1999-07-07,1288,223.4,One Mile,NaN,male,20,0
2,3,3:44.39,Noureddine MORCELI,1970-02-28,ALG,1,Rieti (ITA),1993-09-05,1275,224.3,One Mile,NaN,male,23,0
3,4,3:44.60,Hicham EL GUERROUJ,1974-09-14,MAR,1,Nice (FRA),1998-07-16,1272,224.6,One Mile,NaN,male,23,0
4,5,3:44.90,Hicham EL GUERROUJ,1974-09-14,MAR,1,Oslo (NOR),1997-07-04,1268,224.9,One Mile,NaN,male,22,0


# GRAPHES

## Bar Graph : # athlètes / nat

In [68]:
nationality_counts = df.drop_duplicates(subset='Competitor')['Nat'].value_counts()
nat_counts_dict = nationality_counts.to_dict()

with open("visualizations_data/bar_graph_ath_nat/data.json", "w") as f:
    json.dump(nat_counts_dict, f)

archi pas lisible => 20 most rep countries
urss & russie on merge ?

In [77]:
nationality_counts_20 = nationality_counts.sort_values(ascending=False).head(20)
nat_counts_dict_20 = nationality_counts_20.to_dict()

with open("visualizations_data/bar_graph_ath_nat/data_20.json", "w") as f:
    json.dump(nat_counts_dict_20, f)

## Bar Graph : répartition âge h/f

In [94]:
age_counts = df.groupby(["Sex"])['Age'].value_counts().sort_index()
age_counts_m_dict = age_counts["male"].to_dict()
age_counts_f_dict = age_counts["female"].to_dict()

with open("visualizations_data/bar_chart_age_gen/data_m.json", "w") as f:
    json.dump(age_counts_m_dict, f)
    
with open("visualizations_data/bar_chart_age_gen/data_f.json", "w") as f:
    json.dump(age_counts_f_dict, f)

## Camembert : répartition h/f

In [95]:
gender_counts = df.drop_duplicates(subset='Competitor')['Sex'].value_counts()
gender_counts_dict = gender_counts.to_dict()

with open("visualizations_data/pie_chart_gen/data.json", "w") as f:
    json.dump(gender_counts_dict, f)